# 08 — End-to-end evaluation

**Inputs:** none external — drives `evaluation/harness/run.py` over 100 synthetic multimodal
tickets, generated by `evaluation/datasets/synthetic.py` and submitted through the exact same
`create_ticket` entrypoint `intake_api` uses (UC-8's "same code, different driver" seam).

**Outputs:** `evaluation/reports/e2e_report.{json,csv,md}`

**Runs under `APP_PROFILE=stub`** — no Ollama server or GPU needed in this environment. This is
the load-bearing caveat for reading the numbers below: `StubGenerator` returns one **fixed**
canned diagnosis (`fault_power_supply`, chunk id `c1`) regardless of the ticket, and
`RuleOnlyClassifier` (not the trained sklearn artifact from notebook 02) is forced for CI
determinism. So this run demonstrates **the metrics pipeline is correct and the full pipeline
completes reliably at throughput** — it is not a measurement of real model quality. Re-run with
`APP_PROFILE=cpu` and Ollama running for numbers that reflect the real LLM/classifier.

In [1]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly — the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

cwd: D:\DSEP22\v1 | APP_PROFILE: stub


In [2]:
from pathlib import Path

from evaluation.harness.run import run as run_harness, write_reports

report = run_harness(n=100, database_path=Path("v1_data/harness_nb08.db"), report_dir=Path("evaluation/reports"))
write_reports(report, Path("evaluation/reports"))
report

{'n_tickets': 100,
 'n_completed': 100,
 'n_timed_out': 0,
 'wall_s': 4.5098371505737305,
 'throughput_tickets_per_s': 22.173749663505752,
 'department_accuracy': 0.59,
 'department_macro_f1': 0.5285546605986556,
 'fault_top1_accuracy': 0.22916666666666666,
 'fault_top3_accuracy': 0.22916666666666666,
 'citation_verification_rate': nan,
 'json_validity_rate': 1.0,
 'stage_latency_ms': {'PROCESSING->AGGREGATED': {'n': 100,
   'p50_ms': 2625.2174999999997,
   'p95_ms': 3979.74285,
   'mean_ms': 2426.11357},
  'AGGREGATED->TRIAGED': {'n': 100,
   'p50_ms': 118.059,
   'p95_ms': 363.1236499999998,
   'mean_ms': 150.09167},
  'TRIAGED->DIAGNOSED': {'n': 100,
   'p50_ms': 97.161,
   'p95_ms': 430.80760000000004,
   'mean_ms': 159.15759},
  'DIAGNOSED->READY_FOR_AGENT': {'n': 100,
   'p50_ms': 93.423,
   'p95_ms': 681.34755,
   'mean_ms': 186.44682}}}

## SRS §5.1 stage-latency budget check

In [3]:
for stage, s in report["stage_latency_ms"].items():
    print(f"{stage:30s} p50={s['p50_ms']:8.1f}ms  p95={s['p95_ms']:8.1f}ms  n={s['n']}")

PROCESSING->AGGREGATED         p50=  2625.2ms  p95=  3979.7ms  n=100
AGGREGATED->TRIAGED            p50=   118.1ms  p95=   363.1ms  n=100
TRIAGED->DIAGNOSED             p50=    97.2ms  p95=   430.8ms  n=100
DIAGNOSED->READY_FOR_AGENT     p50=    93.4ms  p95=   681.3ms  n=100


## Reading these numbers honestly

- **Completion rate**: 100/100 tickets reached `READY_FOR_AGENT` — the pipeline itself (fusion,
  state machine, degradation paths) is reliable at this scale under the stub profile.
- **Department accuracy/macro-F1**: reflects `RuleOnlyClassifier`, a small keyword table — not
  the notebook 02 sklearn artifact (test macro-F1 1.0 on held-out Bitext+telecom data). The stub
  profile intentionally never touches the filesystem classifier artifact (CI determinism).
- **Fault top-1/top-3 accuracy**: structurally low because `StubGenerator` returns the *same*
  canned fault for every ticket — only tickets whose expected fault happens to match it can
  score. This metric is only meaningful with a real LLM.
- **Citation verification rate**: NaN — `StubGenerator`'s canned citation (`chunk_id="c1"`) never
  matches a real retrieved chunk id, so `orchestrator_svc.verify_citations` drops it every time
  and no diagnosis ever has a *verified* citation to average over. Also only meaningful with a
  real LLM and an ingested knowledge base.
- **Stage latencies** are real and profile-appropriate: `PROCESSING->AGGREGATED` is the slowest
  stage because it's the one bounded by the 120s aggregation window logic (though it resolves
  immediately here since every attachment's stub adapter succeeds) and by SQLite write
  contention across concurrent broker worker threads — see `09-operations.md` for the production
  (Kafka + Postgres) latency expectations this does not capture.